# 1.3 Comments and Docstrings

## Overview

Documentation is an essential component of professional Python software development. Well-documented code improves maintainability, reproducibility, collaboration, and the reliability of analytical systems.

In quantitative finance and data science, documentation must communicate not only how a program is used but also its mathematical assumptions, input constraints, output conventions, and potential limitations.

This section explores Python comments, docstrings, documentation conventions, runtime introspection, executable examples, and documentation quality assurance.

### Learning Objectives

By the end of this notebook, you will be able to:

1. Distinguish comments from docstrings and understand their respective purposes.
2. Write meaningful comments that explain computational assumptions and design decisions.
3. Apply PEP 8 and PEP 257 documentation conventions.
4. Develop structured docstrings for functions, classes, and modules.
5. Document API contracts, parameters, return values, exceptions, and numerical assumptions.
6. Inspect documentation dynamically using Python's introspection tools.
7. Create executable documentation examples using `doctest`.
8. Identify documentation inconsistencies and common maintenance risks.
9. Develop a professionally documented quantitative finance application.
10. Evaluate documentation quality through automated verification and testing.


## 1.3.1 Comments and Documentation Principles

### Definition

Comments are annotations embedded in source code to explain implementation details, assumptions, constraints, and design decisions.

Python comments begin with the `#` character and are ignored during normal program execution.

### Types of Comments

**1. Block Comments**

Block comments appear on separate lines and typically describe the logic or purpose of the following code.

**2. Inline Comments**

Inline comments appear on the same line as an executable statement.

They should be used sparingly when an implementation requires additional clarification.

**3. Explanatory Comments**

Explanatory comments describe the reasoning behind a particular implementation rather than restating the operation performed by the code.

**4. Assumption and Constraint Comments**

These comments document mathematical assumptions, implementation restrictions, and important analytical decisions.

**5. Maintenance Comments**

Comments may identify technical debt, incomplete implementations, or assumptions requiring further investigation.

Common conventions include `TODO`, `FIXME`, and `NOTE`, although their meanings depend on the project's conventions.

### Fundamental Principle

A useful comment should primarily explain **why** a decision was made, rather than merely describing **what** a readable line of code already expresses.

### Documentation Quality Criteria

High-quality comments should be:

- Accurate and consistent with the implementation.
- Concise without omitting critical context.
- Relevant to non-obvious behavior.
- Maintained alongside source-code changes.
- Explicit about important numerical and financial assumptions.


In [1]:
# ============================================================
# PORTFOLIO EXPOSURE ANALYSIS
# ============================================================

# Portfolio weights are signed:
# Positive weights represent long positions.
# Negative weights represent short positions.

portfolio_weights = {
    "AAPL": 0.55,
    "MSFT": 0.35,
    "TLT": -0.25
}

# Gross exposure measures the total magnitude of
# positions without offsetting long and short exposure.
gross_exposure = sum(
    abs(weight)
    for weight in portfolio_weights.values()
)

# Net exposure preserves position direction and
# measures aggregate directional market exposure.
net_exposure = sum(portfolio_weights.values())

# Leverage ratio is calculated relative to NAV.
# The weights are assumed to be expressed as
# fractions of portfolio net asset value.
leverage_ratio = gross_exposure

# Normalize exposure by gross exposure to show
# the relative risk allocation across positions.
if gross_exposure > 0:
    normalized_exposures = {
        ticker: abs(weight) / gross_exposure
        for ticker, weight in portfolio_weights.items()
    }
else:
    normalized_exposures = {}

print(f"Gross Exposure: {gross_exposure:.2%}")
print(f"Net Exposure: {net_exposure:.2%}")
print(f"Gross Leverage: {leverage_ratio:.2f}x")

print("\nNormalized Exposures:")

for ticker, exposure in normalized_exposures.items():
    print(f"{ticker}: {exposure:.2%}")


Gross Exposure: 115.00%
Net Exposure: 65.00%
Gross Leverage: 1.15x

Normalized Exposures:
AAPL: 47.83%
MSFT: 30.43%
TLT: 21.74%


### Analytical Interpretation

**1. Gross vs. Net Exposure**

Gross exposure represents the sum of absolute portfolio weights, while net exposure measures the signed aggregate exposure.

For a long-short portfolio, these quantities can differ substantially.

**2. Documentation of Financial Assumptions**

The interpretation of gross exposure depends on how portfolio weights are defined.

In this example, weights are expressed relative to portfolio net asset value.

Therefore, the sum of absolute weights represents gross leverage.

**3. Why Comments Matter**

Without documenting the weight convention, another developer might incorrectly interpret the exposure values as fractions of invested capital.

This could lead to incorrect leverage calculations or inconsistent comparisons across portfolios.

**4. General Principle**

In quantitative research, documentation should clarify the mathematical interpretation of each variable, not merely its programming implementation.


## 1.3.2 Docstrings and PEP 257

### Definition

A docstring is a string literal used to document a Python module, function, class, or method.

Docstrings conventionally appear as the first statement within the object being documented.

Unlike ordinary comments, docstrings are generally accessible at runtime through the `__doc__` attribute.

### PEP 257

PEP 257 defines widely accepted conventions for writing Python docstrings.

Its recommendations help developers maintain consistent documentation across Python projects.

### Principal Conventions

**1. Triple Double Quotes**

Use triple double quotes (`"""`) for docstrings, including one-line docstrings.

**2. One-Line Docstrings**

Use a concise summary that describes the object's purpose or behavior.

**3. Multi-Line Docstrings**

A multi-line docstring should contain a summary line, a blank line, and additional explanatory sections.

**4. Consistency**

Documentation should accurately describe the current implementation.

**5. Public Interfaces**

Public modules, classes, functions, and methods should have meaningful documentation.

### Comments vs. Docstrings

| Feature | Comments | Docstrings |
|---|---|---|
| Syntax | `#` | `"""..."""` |
| Primary Purpose | Explain implementation decisions | Document object behavior and interfaces |
| Runtime Access | Not normally available through objects | Accessible through `__doc__` |
| Introspection | Not part of standard object documentation | Supported by `help()` and `inspect` |
| Automated Documentation | Requires source parsing or specialized tools | Supported by common documentation tools |

### Important Distinction

Triple-quoted strings are not automatically docstrings.

A string literal is recognized as an object's docstring only when it appears in the appropriate position within the module, class, or function body.


In [2]:
def calculate_net_exposure(
    portfolio_weights: dict[str, float]
) -> float:
    """
    Calculate the net exposure of a portfolio.

    Parameters
    ----------
    portfolio_weights : dict[str, float]
        Mapping from asset identifiers to signed weights.
        Weights are expressed relative to portfolio NAV.

    Returns
    -------
    float
        Sum of signed portfolio weights.

    Notes
    -----
    Positive weights represent long positions.
    Negative weights represent short positions.

    This function does not validate whether the
    supplied portfolio weights are economically valid.

    Examples
    --------
    >>> calculate_net_exposure(
    ...     {"AAPL": 0.6, "MSFT": 0.4, "TLT": -0.2}
    ... )
    0.8
    """
    return sum(portfolio_weights.values())


weights = {
    "AAPL": 0.60,
    "MSFT": 0.40,
    "TLT": -0.20
}

result = calculate_net_exposure(weights)

print(f"Net Exposure: {result:.2%}")


Net Exposure: 80.00%


## 1.3.3 Structured Docstrings and API Contracts

### Documentation as an API Contract

An API contract describes how a function or component is expected to behave.

In professional software development, a function's documentation should communicate its inputs, outputs, assumptions, failure conditions, and externally observable behavior.

A docstring describes the intended contract, but it does not automatically enforce that contract at runtime.

### Common Docstring Components

**1. Summary**

A concise explanation of what the function does.

**2. Parameters**

Input arguments, expected types, meanings, units, and restrictions.

**3. Returns**

The returned value, its type, and its interpretation.

**4. Raises**

Exceptions that may occur under documented conditions.

**5. Notes**

Mathematical definitions, implementation assumptions, numerical considerations, and limitations.

**6. Examples**

Executable or illustrative examples showing expected behavior.

**7. See Also**

References to related functions or relevant documentation.

### Common Documentation Styles

**NumPy Style**

Uses section headings such as `Parameters`, `Returns`, `Raises`, and `Notes`.

It is widely used in scientific computing.

**Google Style**

Uses sections such as `Args:`, `Returns:`, and `Raises:`.

It is common in general-purpose Python applications.

**reStructuredText Style**

Uses field lists such as `:param:`, `:type:`, and `:returns:`.

It is frequently used with Sphinx documentation.

### Recommended Convention

For this quantitative finance and data science project, use NumPy-style docstrings consistently.

### Critical Financial Documentation Requirements

Financial functions should explicitly document:

- Whether returns are decimal values or percentages.
- Whether volatility is daily, monthly, or annualized.
- Whether rates are simple or continuously compounded.
- Whether portfolio weights are based on NAV or gross exposure.
- Whether cash and transaction costs are included.
- Whether calculations use sample or population statistics.
- Whether numerical results rely on floating-point approximations.
- How invalid inputs and missing observations are handled.


In [3]:
from math import isfinite


def annualized_growth_rate(
    initial_value: float,
    final_value: float,
    years: float
) -> float:
    """
    Calculate the compound annual growth rate.

    Parameters
    ----------
    initial_value : float
        Initial investment value, strictly positive.
    final_value : float
        Final investment value, strictly positive.
    years : float
        Investment horizon in years, strictly positive.

    Returns
    -------
    float
        Annualized compound growth rate in decimal form.

    Raises
    ------
    ValueError
        If any input is nonfinite or nonpositive.
    TypeError
        If an input is not a real numeric value.

    Notes
    -----
    CAGR is defined as:

        CAGR = (V_final / V_initial) ** (1 / T) - 1

    This measure assumes a positive initial and final
    value and does not account for intermediate cash flows.

    Examples
    --------
    >>> f"{annualized_growth_rate(100, 121, 2):.2%}"
    '10.00%'

    >>> f"{annualized_growth_rate(100, 81, 2):.2%}"
    '-10.00%'
    """
    inputs = (initial_value, final_value, years)

    for value in inputs:
        if isinstance(value, bool) or not isinstance(
            value, (int, float)
        ):
            raise TypeError("Inputs must be real numbers.")

        if not isfinite(value) or value <= 0:
            raise ValueError(
                "Inputs must be finite and positive."
            )

    return (
        (final_value / initial_value) ** (1 / years)
        - 1
    )


cagr = annualized_growth_rate(
    initial_value=10000,
    final_value=14641,
    years=4
)

print(f"Annualized Growth Rate: {cagr:.2%}")


Annualized Growth Rate: 10.00%


## 1.3.4 Runtime Introspection and Documentation Discovery

### Runtime Introspection

Introspection is the ability of a program to examine the properties, structure, and metadata of objects during execution.

Python provides several mechanisms for examining function signatures, documentation strings, type annotations, and other object attributes.

### Core Introspection Tools

**1. `__doc__`**

Accesses an object's documentation string when available.

**2. `help()`**

Displays interactive documentation for Python objects.

**3. `inspect.getdoc()`**

Retrieves an object's documentation and normalizes its indentation.

**4. `inspect.signature()`**

Retrieves a callable's parameter and return-annotation structure when available.

**5. `__annotations__`**

Provides the object's stored type annotations.

**6. `inspect.getsource()`**

Attempts to retrieve the source code of an object.

### Advanced Considerations

- Type annotations are not automatically enforced by Python.
- Docstrings do not guarantee correctness.
- Runtime introspection may fail for dynamically generated objects or implementations without accessible source code.
- Documentation inherited from parent classes may differ from an overriding method's behavior.
- Wrappers and decorators may affect accessible signatures and metadata.

### Quantitative Research Applications

Introspection can support automatic API documentation, validation tooling, experiment reporting, and research-library inspection.


In [4]:
import inspect

# Retrieve the function signature
signature = inspect.signature(
    annualized_growth_rate
)

# Retrieve the normalized docstring
documentation = inspect.getdoc(
    annualized_growth_rate
)

# Retrieve type annotations
annotations = annualized_growth_rate.__annotations__

print("FUNCTION SIGNATURE")
print(signature)

print("\nTYPE ANNOTATIONS")
print(annotations)

print("\nDOCUMENTATION")
print(documentation)

# Check whether the function has documentation
assert documentation is not None

# Verify the expected documentation sections
required_sections = [
    "Parameters",
    "Returns",
    "Raises",
    "Notes",
    "Examples"
]

for section in required_sections:
    assert section in documentation

print("\nDocumentation structure verified.")


FUNCTION SIGNATURE
(initial_value: float, final_value: float, years: float) -> float

TYPE ANNOTATIONS
{'initial_value': <class 'float'>, 'final_value': <class 'float'>, 'years': <class 'float'>, 'return': <class 'float'>}

DOCUMENTATION
Calculate the compound annual growth rate.

Parameters
----------
initial_value : float
    Initial investment value, strictly positive.
final_value : float
    Final investment value, strictly positive.
years : float
    Investment horizon in years, strictly positive.

Returns
-------
float
    Annualized compound growth rate in decimal form.

Raises
------
ValueError
    If any input is nonfinite or nonpositive.
TypeError
    If an input is not a real numeric value.

Notes
-----
CAGR is defined as:

    CAGR = (V_final / V_initial) ** (1 / T) - 1

This measure assumes a positive initial and final
value and does not account for intermediate cash flows.

Examples
--------
>>> f"{annualized_growth_rate(100, 121, 2):.2%}"
'10.00%'

>>> f"{annualized_grow

## 1.3.5 Executable Documentation and Doctest

### Executable Documentation

Executable documentation contains examples that can be evaluated programmatically.

Python provides the `doctest` module to identify interactive examples embedded in docstrings and verify their expected outputs.

### How Doctest Works

1. Identify interactive Python examples beginning with `>>>`.
2. Execute the examples in a suitable namespace.
3. Capture their resulting output.
4. Compare the output with the documented expectation.
5. Report differences between actual and expected behavior.

### Advantages

- Detects outdated documentation examples.
- Provides basic behavioral regression checks.
- Encourages concise and reproducible usage examples.
- Connects API documentation with executable code.

### Important Limitations

- Doctest is not a substitute for comprehensive unit testing.
- Output comparisons can be sensitive to formatting.
- Floating-point representations may introduce fragile comparisons.
- Examples depending on external files or network resources can be difficult to reproduce.
- Doctest coverage is limited to the behavior exercised by the examples.

### Recommended Practice

Use doctest for simple, deterministic examples and separate unit tests for extensive edge-case coverage.

For numerical examples, explicitly format floating-point output or use robust numerical assertions.


In [5]:
import doctest


def maximum_drawdown(
    equity_curve: list[float]
) -> float:
    """
    Calculate the maximum drawdown of an equity curve.

    Parameters
    ----------
    equity_curve : list[float]
        Nonempty sequence of strictly positive,
        finite portfolio equity observations.

    Returns
    -------
    float
        Maximum drawdown expressed as a nonpositive
        decimal return.

    Raises
    ------
    ValueError
        If the input is empty or contains an invalid
        equity observation.

    Notes
    -----
    The running peak is updated chronologically.

    Drawdown at time t is defined as:

        D_t = Equity_t / RunningPeak_t - 1

    Maximum drawdown is the minimum observed D_t.

    Examples
    --------
    >>> f"{maximum_drawdown([100, 120, 90, 108]):.2%}"
    '-25.00%'

    >>> maximum_drawdown([100, 110, 120])
    0.0
    """
    if not equity_curve:
        raise ValueError("Equity curve cannot be empty.")

    for value in equity_curve:
        if (
            isinstance(value, bool)
            or not isinstance(value, (int, float))
            or not isfinite(value)
            or value <= 0
        ):
            raise ValueError("Equity values must be positive.")

    running_peak = equity_curve[0]
    worst_drawdown = 0.0

    for equity in equity_curve:
        running_peak = max(running_peak, equity)

        drawdown = equity / running_peak - 1

        worst_drawdown = min(
            worst_drawdown,
            drawdown
        )

    return worst_drawdown


# Execute examples from the function's docstring.
doctest.run_docstring_examples(
    maximum_drawdown,
    globals(),
    name="maximum_drawdown",
    verbose=True
)


Finding tests in maximum_drawdown
Trying:
    f"{maximum_drawdown([100, 120, 90, 108]):.2%}"
Expecting:
    '-25.00%'
ok
Trying:
    maximum_drawdown([100, 110, 120])
Expecting:
    0.0
ok


## 1.3.6 Documentation Quality and Common Pitfalls

### 1. Redundant Comments

Comments that merely repeat obvious operations increase maintenance overhead without improving understanding.

### 2. Outdated Documentation

Documentation becomes misleading when the implementation changes but its descriptions and examples remain unchanged.

### 3. Ambiguous Units

Financial variables such as volatility, returns, interest rates, and exposures must have explicit units and conventions.

### 4. Undocumented Edge Cases

Functions should document important restrictions, including zero denominators, empty inputs, invalid values, and unsupported assumptions.

### 5. Missing Numerical Assumptions

Quantitative functions may depend on annualization conventions, sampling frequencies, interpolation rules, or floating-point approximations.

These assumptions should be documented when they affect interpretation.

### 6. Misleading Type Information

Type hints and docstrings should agree with the actual accepted inputs and returned outputs.

Type annotations alone do not perform runtime validation.

### 7. Documentation Duplication

Duplicating lengthy explanations across comments, docstrings, README files, and notebooks can create inconsistencies.

Documentation should be organized so that each piece of information has a clearly defined purpose.

### 8. Overdocumentation

Excessive comments can obscure the implementation and make the code harder to maintain.

Prioritize information that is necessary to understand behavior, assumptions, and design decisions.

### Production Documentation Checklist

- Public interfaces have accurate docstrings.
- Parameters and return values are correctly documented.
- Numerical units and assumptions are explicit.
- Important exceptions are documented.
- Examples are reproducible.
- Documentation matches the actual implementation.
- Internal comments explain non-obvious decisions.
- Automated tests validate documented behavior.


## 1.3.7 Practical Application: Documentation-Driven Portfolio Stress Testing API

### Objective

Design a documented Python API for deterministic portfolio stress testing.

The system should provide clearly defined data structures, numerical assumptions, error handling, and reproducible examples.

### Functional Requirements

1. Represent individual portfolio positions using structured objects.
2. Validate position quantities, prices, and classifications.
3. Calculate portfolio market value.
4. Apply sector-specific price shocks.
5. Calculate stressed portfolio value and scenario P&L.
6. Calculate portfolio-level stress returns.
7. Attribute scenario P&L to individual positions.
8. Identify the largest negative P&L contributor.
9. Return results using a structured data model.
10. Validate documented examples using automated tests.

### Documentation Requirements

Every public component should describe:

- Its purpose and expected behavior.
- Input parameters and their types.
- Relevant numerical units.
- Return values and their interpretation.
- Possible exceptions.
- Financial assumptions and limitations.
- Executable examples where appropriate.

### Financial Assumptions

- The portfolio contains long-only positions.
- Cash balances are nonnegative.
- Market prices are expressed in USD.
- Sector shocks represent instantaneous percentage changes.
- Cash is not affected by market shocks.
- Transaction costs, dividends, and financing costs are excluded.
- Results use floating-point arithmetic and are not intended for accounting-grade settlement calculations.

### Implementation Constraints

- Python standard library only.
- No NumPy or Pandas.
- No external financial data.
- No mutation of the original position objects.
- Reproducible calculations and deterministic examples.


In [6]:
# ============================================================
# DOCUMENTATION-DRIVEN PORTFOLIO STRESS TESTING API
# ============================================================

from dataclasses import dataclass
from collections.abc import Mapping, Sequence
from math import isfinite
import doctest
import inspect


@dataclass(frozen=True)
class Position:
    """
    Represent a long-only security position.

    Parameters
    ----------
    ticker : str
        Nonempty asset identifier, unique within
        the portfolio being evaluated.
    sector : str
        Sector classification used for stress testing.
    shares : int
        Strictly positive number of shares.
    price : float
        Finite, strictly positive market price in USD.

    Raises
    ------
    ValueError
        If an identifier, quantity, or price is invalid.

    Notes
    -----
    The position is modeled as a frozen dataclass.
    This example supports integer quantities only.
    """

    ticker: str
    sector: str
    shares: int
    price: float

    def __post_init__(self) -> None:
        """Validate the position after initialization."""

        if not isinstance(self.ticker, str) or not self.ticker.strip():
            raise ValueError("ticker must be a nonempty string")

        if not isinstance(self.sector, str) or not self.sector.strip():
            raise ValueError("sector must be a nonempty string")

        if (
            isinstance(self.shares, bool)
            or not isinstance(self.shares, int)
            or self.shares <= 0
        ):
            raise ValueError("shares must be a positive integer")

        if (
            isinstance(self.price, bool)
            or not isinstance(self.price, (int, float))
            or not isfinite(self.price)
            or self.price <= 0
        ):
            raise ValueError("price must be finite and positive")


@dataclass
class StressResult:
    """
    Store portfolio stress-testing results.

    Attributes
    ----------
    base_value : float
        Initial portfolio value, including cash, in USD.
    stressed_value : float
        Portfolio value after market shocks, in USD.
    pnl : float
        Portfolio dollar gain or loss from the scenario.
    return_pct : float
        Scenario return in decimal form.
    contributions : dict[str, float]
        Dollar P&L contributions by asset ticker.
    worst_contributor : str or None
        Asset with the most negative P&L contribution.
        None indicates no negative contributions.
    """

    base_value: float
    stressed_value: float
    pnl: float
    return_pct: float
    contributions: dict[str, float]
    worst_contributor: str | None


def stress_portfolio(
    positions: Sequence[Position],
    sector_shocks: Mapping[str, float],
    *,
    cash: float = 0.0,
) -> StressResult:
    """
    Apply deterministic sector shocks to a portfolio.

    Parameters
    ----------
    positions : Sequence[Position]
        Nonempty sequence of long-only positions.
        All asset tickers must be unique.
    sector_shocks : Mapping[str, float]
        Sector shocks expressed as decimal returns.
        A value of -0.10 represents a 10% price decline.
        Every portfolio sector must have a defined shock.
    cash : float, default 0.0
        Finite, nonnegative cash balance in USD.

    Returns
    -------
    StressResult
        Structured portfolio valuation and risk results.

    Raises
    ------
    ValueError
        If positions are empty, tickers are duplicated,
        cash is invalid, or a shock is nonfinite
        or below -100%.
    KeyError
        If a required sector shock is missing.
    TypeError
        If a holding is not a Position instance.

    Notes
    -----
    Each asset receives the shock assigned to its sector.

    Asset P&L is defined as:

        PnL_i = MarketValue_i * SectorShock_i

    Portfolio scenario return is defined as:

        R = Sum(PnL_i) / InitialPortfolioValue

    InitialPortfolioValue includes cash.

    Cash remains unchanged during the stress scenario.

    The model is deterministic and excludes transaction
    costs, liquidity constraints, and dynamic correlations.

    The calculations use floating-point arithmetic and
    are intended for illustrative analytical purposes.

    Examples
    --------
    >>> holdings = [
    ...     Position("AAA", "Tech", 10, 100.0),
    ...     Position("BBB", "Bonds", 5, 80.0),
    ... ]
    >>> result = stress_portfolio(
    ...     holdings, {"Tech": -0.10, "Bonds": 0.05}, cash=100.0
    ... )
    >>> f"{result.pnl:.2f}"
    '-80.00'
    >>> f"{result.return_pct:.4%}"
    '-5.3333%'
    >>> result.worst_contributor
    'AAA'
    """

    if not positions:
        raise ValueError("positions must not be empty")

    if (
        isinstance(cash, bool)
        or not isinstance(cash, (int, float))
        or not isfinite(cash)
        or cash < 0
    ):
        raise ValueError("cash must be finite and nonnegative")

    contributions = {}
    invested_value = 0.0

    for position in positions:

        if not isinstance(position, Position):
            raise TypeError(
                "all holdings must be Position instances"
            )

        if position.ticker in contributions:
            raise ValueError(
                f"duplicate ticker: {position.ticker}"
            )

        if position.sector not in sector_shocks:
            raise KeyError(
                f"missing shock for sector: {position.sector}"
            )

        shock = sector_shocks[position.sector]

        if (
            isinstance(shock, bool)
            or not isinstance(shock, (int, float))
            or not isfinite(shock)
            or shock < -1.0
        ):
            raise ValueError(
                f"invalid shock for sector: {position.sector}"
            )

        market_value = position.shares * position.price

        # Sector shocks apply to invested holdings.
        # Cash does not experience a price shock.
        position_pnl = market_value * shock

        invested_value += market_value
        contributions[position.ticker] = position_pnl

    base_value = invested_value + cash

    pnl = sum(contributions.values())

    # Identify the largest negative contribution.
    # A portfolio without losses has no loss contributor.
    worst_ticker = min(
        contributions,
        key=contributions.get
    )

    if contributions[worst_ticker] >= 0:
        worst_ticker = None

    return StressResult(
        base_value=base_value,
        stressed_value=base_value + pnl,
        pnl=pnl,
        return_pct=pnl / base_value,
        contributions=contributions,
        worst_contributor=worst_ticker
    )


In [7]:
# ============================================================
# PORTFOLIO STRESS TESTING EXAMPLE
# ============================================================

positions = [
    Position("AAPL", "Technology", 50, 190.0),
    Position("MSFT", "Technology", 20, 410.0),
    Position("JPM", "Financials", 40, 205.0),
    Position("TLT", "Bonds", 80, 92.0)
]

sector_shocks = {
    "Technology": -0.20,
    "Financials": -0.10,
    "Bonds": 0.05
}

result = stress_portfolio(
    positions=positions,
    sector_shocks=sector_shocks,
    cash=5000.0
)

print("=" * 55)
print("PORTFOLIO STRESS TEST REPORT")
print("=" * 55)

print(f"Initial Value: ${result.base_value:,.2f}")
print(f"Stressed Value: ${result.stressed_value:,.2f}")
print(f"Scenario P&L: ${result.pnl:,.2f}")
print(f"Scenario Return: {result.return_pct:.2%}")

print("\nPOSITION CONTRIBUTIONS")
print("-" * 55)

for ticker, contribution in result.contributions.items():
    print(f"{ticker:<8} ${contribution:>12,.2f}")

print("\nLARGEST LOSS CONTRIBUTOR")
print("-" * 55)

print(result.worst_contributor)


PORTFOLIO STRESS TEST REPORT
Initial Value: $38,260.00
Stressed Value: $34,268.00
Scenario P&L: $-3,992.00
Scenario Return: -10.43%

POSITION CONTRIBUTIONS
-------------------------------------------------------
AAPL     $   -1,900.00
MSFT     $   -1,640.00
JPM      $     -820.00
TLT      $      368.00

LARGEST LOSS CONTRIBUTOR
-------------------------------------------------------
AAPL


In [8]:
# ============================================================
# DOCUMENTATION QUALITY ASSURANCE
# ============================================================

public_objects = [
    Position,
    StressResult,
    stress_portfolio
]

required_sections = {
    Position: ["Parameters"],
    StressResult: ["Attributes"],
    stress_portfolio: [
        "Parameters",
        "Returns",
        "Raises",
        "Notes",
        "Examples"
    ]
}

print("DOCUMENTATION AUDIT")
print("-" * 55)

for obj in public_objects:

    documentation = inspect.getdoc(obj)

    assert documentation is not None, (
        f"Missing documentation: {obj.__name__}"
    )

    for section in required_sections[obj]:
        assert section in documentation, (
            f"Missing '{section}' in {obj.__name__}"
        )

    print(f"{obj.__name__}: Documentation verified")


print("\nRUNNING EXECUTABLE DOCUMENTATION TESTS")
print("-" * 55)

doctest.run_docstring_examples(
    stress_portfolio,
    globals(),
    name="stress_portfolio",
    verbose=True
)


DOCUMENTATION AUDIT
-------------------------------------------------------
Position: Documentation verified
StressResult: Documentation verified
stress_portfolio: Documentation verified

RUNNING EXECUTABLE DOCUMENTATION TESTS
-------------------------------------------------------
Finding tests in stress_portfolio
Trying:
    holdings = [
        Position("AAA", "Tech", 10, 100.0),
        Position("BBB", "Bonds", 5, 80.0),
    ]
Expecting nothing
ok
Trying:
    result = stress_portfolio(
        holdings, {"Tech": -0.10, "Bonds": 0.05}, cash=100.0
    )
Expecting nothing
ok
Trying:
    f"{result.pnl:.2f}"
Expecting:
    '-80.00'
ok
Trying:
    f"{result.return_pct:.4%}"
Expecting:
    '-5.3333%'
ok
Trying:
    result.worst_contributor
Expecting:
    'AAA'
ok


### Analytical Interpretation

**1. Documentation as an Interface Contract**

The portfolio stress-testing API uses structured docstrings to define how its components should be used.

Each public component documents its expected inputs, outputs, and relevant assumptions.

**2. Numerical Transparency**

The documentation explicitly defines the sector shock convention, the calculation of scenario P&L, and the treatment of cash.

These definitions are essential for interpreting financial results consistently.

**3. Input Validation**

The implementation validates several important input constraints, including positive prices, positive share quantities, unique asset identifiers, and valid sector shocks.

Documented exceptions clarify how invalid inputs are handled.

**4. Reproducibility**

Executable docstring examples demonstrate the expected behavior of the stress-testing function.

Automated verification helps detect inconsistencies between documentation and implementation.

**5. Software Maintainability**

Structured documentation and reusable data models improve the readability and maintainability of quantitative research software.

The separation between position representation, portfolio calculations, and reporting supports future extensions.

**6. Limitations**

The stress-testing engine uses deterministic sector shocks and does not estimate loss probabilities or dynamic asset correlations.

It also uses floating-point arithmetic and does not implement a complete production-grade numerical validation framework.

**7. Further Development**

Potential improvements include:

- More comprehensive numerical validation.
- Automated unit and property-based testing.
- Documentation generation with Sphinx.
- Integration with continuous integration pipelines.
- Historical and probabilistic scenario analysis.
- Support for additional instrument types.


## Advanced Practice Exercises

### Exercise 1: Professional API Documentation

**Objective**

Develop comprehensive documentation for a financial analytics library.

**Tasks**

1. Implement three functions:
   - `calculate_returns()`
   - `annualized_volatility()`
   - `maximum_drawdown()`
2. Write NumPy-style docstrings for every function.
3. Document all input parameters and return values.
4. Specify numerical units and mathematical assumptions.
5. Document invalid input conditions and expected exceptions.
6. Provide at least two executable examples per function.
7. Verify the documentation using runtime introspection.

**Requirements**

- Use consistent documentation conventions.
- Distinguish daily returns from annualized measurements.
- Specify the standard deviation convention used.
- Explain the treatment of missing or invalid observations.
- Keep documentation synchronized with the implementation.

---

### Exercise 2: Documentation Consistency and Contract Testing

**Objective**

Identify and correct inconsistencies between function documentation and actual behavior.

**Tasks**

1. Create five functions with deliberately incorrect docstrings.
2. Include incorrect descriptions of:
   - Return types
   - Parameter constraints
   - Numerical units
   - Exception behavior
   - Mathematical assumptions
3. Develop tests that identify behavioral inconsistencies.
4. Correct each docstring or implementation.
5. Explain why each inconsistency could cause incorrect downstream analysis.

**Requirements**

- Do not rely exclusively on string matching.
- Include behavioral tests.
- Test valid and invalid inputs.
- Document the corrected function contracts.

---

### Exercise 3: Automated Documentation Quality Auditor

**Objective**

Develop a Python utility that evaluates documentation quality across a collection of functions and classes.

**Tasks**

1. Inspect public functions and classes dynamically.
2. Identify objects without docstrings.
3. Detect missing required documentation sections.
4. Compare function signatures with documented parameter names.
5. Identify undocumented return values.
6. Detect potentially undocumented exceptions.
7. Execute available doctest examples.
8. Generate a structured documentation audit report.

**Requirements**

- Use Python's standard library.
- Support functions with multiple parameters.
- Handle optional and keyword-only arguments.
- Distinguish missing documentation from incorrect documentation.
- Explain the limitations of static documentation analysis.

---

### Exercise 4: Financial Documentation Reproducibility

**Objective**

Build a reproducible documentation system for a quantitative research experiment.

**Tasks**

1. Implement a portfolio risk calculation.
2. Document all model assumptions.
3. Record input parameters and their units.
4. Document calculation formulas.
5. Include deterministic examples.
6. Verify results against independently calculated reference values.
7. Test sensitivity to invalid inputs.
8. Generate a reproducibility report.

**Requirements**

- No external data dependencies.
- Explicitly document random seeds if randomness is introduced.
- Document floating-point precision considerations.
- Avoid undocumented global-state dependencies.
- Ensure examples are independently executable.

---

### Exercise 5: Advanced Documentation Refactoring

**Objective**

Refactor a poorly documented financial analytics program into a reusable and maintainable research library.

**Tasks**

1. Identify redundant and misleading comments.
2. Replace ambiguous variable descriptions with explicit numerical conventions.
3. Introduce structured function and class docstrings.
4. Document preconditions and postconditions.
5. Separate public interface documentation from implementation comments.
6. Add executable usage examples.
7. Verify documentation consistency after refactoring.
8. Produce a documentation quality assessment.

**Requirements**

- Preserve the original financial calculations.
- Follow PEP 8 and PEP 257 conventions.
- Use consistent naming and formatting.
- Avoid unnecessary documentation duplication.
- Include automated regression tests.

---

## Advanced Challenge: Documentation-Driven Quantitative Research Library

**Objective**

Develop a professionally documented quantitative finance package with automated documentation validation.

**System Components**

1. Financial data validation
2. Return calculations
3. Risk metrics
4. Portfolio exposure analysis
5. Stress testing
6. Performance attribution
7. Documentation auditing

**Technical Requirements**

- Python standard library only.
- Modular software architecture.
- NumPy-style docstrings.
- Explicit type annotations.
- Documented exception behavior.
- Executable docstring examples.
- Automated unit tests.
- Documentation coverage reporting.
- Reproducible numerical calculations.
- Consistent mathematical conventions.

**Additional Challenges**

1. Detect mismatches between function signatures and documented parameters.
2. Test documentation examples automatically.
3. Develop a documentation completeness score.
4. Identify documentation changes that may require API compatibility reviews.
5. Integrate documentation validation into a GitHub Actions workflow.

**Expected Deliverables**

1. Documented Python source code.
2. API reference documentation.
3. Automated doctest results.
4. Unit and integration tests.
5. Documentation quality report.
6. Technical discussion of assumptions, limitations, and potential improvements.
